# ResNet18 ERPgnostics training export

This notebook trains the binary ERP-pattern ResNet18 on the real JLD2 datasets
under `datasets/` and writes a reusable model artifact. It intentionally does
not build the ERPgnostics explorer; use
`resnet18_erpgnostics_import_explorer.ipynb` for visualization.



In [1]:
# Optional quick-run overrides. Uncomment before running this first cell.
# ENV["WEEK24_RUN_CV"] = "false"
# ENV["WEEK24_RESNET18_EPOCHS"] = "1"

include(joinpath(@__DIR__, "resnet18_erpgnostics_common.jl"))

TRAIN_EXPORT_DIR = joinpath(NOTEBOOK_DIR, "outputs", "resnet18_erpgnostics_train_export")
TRAIN_MODEL_ARTIFACT = model_artifact_path(TRAIN_EXPORT_DIR)
EXPORT_PARENT_SCORES = lowercase(get(ENV, "WEEK24_EXPORT_PARENT_SCORES", "true")) in ("true", "1", "yes")

println("Training output directory: ", TRAIN_EXPORT_DIR)
println("Model artifact path: ", TRAIN_MODEL_ARTIFACT)
println("TARGET_TRIALS = ", TARGET_TRIALS)
println("EXPORT_PARENT_SCORES = ", EXPORT_PARENT_SCORES)



REPO_ROOT = /home/benjamin/Dokumente/BA2/
DATASETS_ROOT = /home/benjamin/Dokumente/BA2/datasets
Training output directory: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export
Model artifact path: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export/resnet18_final_state.jld2
TARGET_TRIALS = 200
EXPORT_PARENT_SCORES = true


## Train

By default this runs the same 5-fold CV plus final all-data model as the
combined notebook. For a quick smoke run, set these before executing:

```julia
ENV["WEEK24_RUN_CV"] = "false"
ENV["WEEK24_RESNET18_EPOCHS"] = "1"
```



In [2]:
training_run = run_training_pipeline(
    output_dir = TRAIN_EXPORT_DIR,
    nepochs = TRAIN_EPOCHS,
    lr = TRAIN_LR,
    k_folds = K_FOLDS,
    seed = GLOBAL_SEED,
    run_cv = RUN_CV,
)



Loading real JLD2 labels from /home/benjamin/Dokumente/BA2/datasets.
Materializing fixed-trial ERP images with inverse-sort/polarity variants.
Training and validating ResNet18 with 5-fold CV.
CUDA device: NVIDIA GeForce RTX 4070
resnet18_real_jld2_inverse_sort_polarity_binary | CV fold 1/5 | train=14876 | val=3720
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 1/8 | loss=0.52504
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 2/8 | loss=0.28386
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 3/8 | loss=0.19982
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 4/8 | loss=0.16272
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 5/8 | loss=0.13827
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 6/8 | loss=0.15003
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 7/8 | loss=0.09704
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 8/8 | loss=0.0767
resnet18_real_jld2_inverse_sort_polar

(labels_df = 2869×9 DataFrame
  Row │ dataset_key                dataset_label                      channel_ ⋯
      │ String                     String                             String   ⋯
──────┼─────────────────────────────────────────────────────────────────────────
    1 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E1       ⋯
    2 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E111
    3 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E121
    4 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E51
    5 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E52      ⋯
    6 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E59
    7 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E70
    8 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E72
  ⋮   │             ⋮                              ⋮                       ⋮   ⋱
 2863 │ nod_eeg_public             NOD

## Export Model

The artifact stores the full CPU `Flux.state(model)` plus metadata. This is
important for ResNet18 because BatchNorm running statistics are model state,
not trainable parameters. Saving only `Flux.trainables(model)` makes the
imported model produce nearly constant scores.



In [3]:
artifact_metadata = Dict{String, Any}(
    "source_notebook" => "notebooks/week_24/resnet18_erpgnostics_train_export.ipynb",
    "output_dir" => TRAIN_EXPORT_DIR,
    "run_config_path" => joinpath(TRAIN_EXPORT_DIR, "run_config.json"),
    "final_train_metrics_path" => joinpath(TRAIN_EXPORT_DIR, "final_train_metrics.csv"),
    "fold_metrics_path" => joinpath(TRAIN_EXPORT_DIR, "fold_metrics.csv"),
    "trained_on" => "real JLD2 datasets from datasets/, excluding datasets/simulated",
    "all_parent_scores_path" => parent_scores_path(TRAIN_EXPORT_DIR),
    "all_augmentation_scores_path" => augmentation_scores_path(TRAIN_EXPORT_DIR),
)

save_resnet18_model_artifact(TRAIN_MODEL_ARTIFACT, training_run.final_model; metadata = artifact_metadata)
println("Saved model artifact: ", TRAIN_MODEL_ARTIFACT)

training_run.final.metrics_df



Saved model artifact: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export/resnet18_final_state.jld2


Row,model_name,n_train,train_accuracy,train_balanced_accuracy,train_macro_f1,train_precision,train_recall,train_time_s,pretrained_params_loaded,batchsize,use_cuda
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Int64,Int64,Bool
1,resnet18_real_jld2_inverse_sort_polarity_binary_final,18596,0.978167,0.979541,0.977978,0.976863,0.979541,35.4721,62,32,true


## Export Model Scores

This writes one parent probability per dataset/sort-variable/channel ERP image
for all real datasets under `datasets/`, excluding `datasets/simulated`.
Sort variables are included only when the dataset has at least one manual
pattern label for that sort variable.



In [4]:
if EXPORT_PARENT_SCORES
    println("Scoring all labelled-sort parent ERP images with the final ResNet18.")
    parent_score_run = score_dataset_parent_images(
        training_run.final_model,
        training_run.final.device;
        labels_df = training_run.labels_df,
    )
    saved_score_paths = save_parent_score_outputs(TRAIN_EXPORT_DIR, parent_score_run)
    println("Saved parent scores: ", saved_score_paths.parent_scores_path)
    println("Saved augmentation scores: ", saved_score_paths.augmentation_scores_path)
    if nrow(parent_score_run.skipped_df) > 0
        @warn "Some dataset/sort/channel combinations could not be scored." skipped_rows = nrow(parent_score_run.skipped_df)
    end
    first(parent_score_run.score_df, min(12, nrow(parent_score_run.score_df)))
else
    println("Skipping parent-score export because WEEK24_EXPORT_PARENT_SCORES=false.")
    parent_score_run = nothing
end



Scoring all labelled-sort parent ERP images with the final ResNet18.
Saved parent scores: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export/all_parent_scores.csv
Saved augmentation scores: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export/all_augmentation_scores.csv


Row,parent_image_id,dataset_key,dataset_label,channel_name,channel_idx,sort_variable,true_erp_class,true_binary_label,has_manual_label,n_manual_labels,n_manual_pattern_labels,score_class,score_class_std,score_class_min,score_class_max,n_augmentations
,String,String,String,String,Int64,String,String,Int64,Bool,Int64,Int64,Float32,Float32,Float32,Float32,Int64
1,02_new_eegeyenet_saccades::E1::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E1,1,saccade_duration,no_class,0,true,1,0,0.00126035,0.00155745,3.23521e-6,0.00321539,4
2,02_new_eegeyenet_saccades::E2::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E2,2,saccade_duration,no_class,0,true,1,0,0.0173106,0.0176785,0.00510723,0.0435176,4
3,02_new_eegeyenet_saccades::E3::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E3,3,saccade_duration,no_class,0,true,1,0,0.00606515,0.0083905,0.00029697,0.0183745,4
4,02_new_eegeyenet_saccades::E4::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E4,4,saccade_duration,no_class,0,true,1,0,0.335099,0.333175,0.0341527,0.811181,4
5,02_new_eegeyenet_saccades::E5::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E5,5,saccade_duration,no_class,0,true,1,0,0.00681957,0.0129321,6.02899e-5,0.0262096,4
6,02_new_eegeyenet_saccades::E6::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E6,6,saccade_duration,no_class,0,true,1,0,3.02736e-7,3.66982e-7,1.6844e-8,7.94664e-7,4
7,02_new_eegeyenet_saccades::E7::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E7,7,saccade_duration,no_class,0,true,1,0,1.50869e-5,2.00618e-5,7.0658e-9,4.29876e-5,4
8,02_new_eegeyenet_saccades::E8::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E8,8,saccade_duration,no_class,0,true,1,0,0.25186,0.372692,0.00501795,0.794221,4
9,02_new_eegeyenet_saccades::E9::saccade_duration::full_parent,02_new_eegeyenet_saccades,EEGEyeNet minimally processed saccades,E9,9,saccade_duration,no_class,0,true,1,0,0.221946,0.297847,0.016144,0.659999,4


## Training Summary



In [5]:
if nrow(training_run.cv.metrics_df) > 0
    summarize_metrics(training_run.cv.metrics_df)
else
    training_run.final.metrics_df
end


Row,model_name,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_precision_mean,val_recall_mean,train_time_mean_s,pretrained_params_loaded
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64
1,resnet18_real_jld2_inverse_sort_polarity_binary,0.911217,0.00893801,0.910859,0.00893857,0.910266,0.00897813,0.910369,0.910859,36.98,62
